# Chapter 1 – Loading and Visualising Drone Data

This Jupyter notebook loads a 10-band **MicaSense RedEdge-P Dual** image, summarises its
spectral bands, creates true- and false-colour composites, and places
the drone image on an interactive basemap.

The imagery is read directly from the workshop **STAC catalogue** over the
internet, so there is nothing to download. Run the cells in order from the top.

## 1. Load libraries and shared workshop functions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import ipywidgets as widgets
import folium
import pystac
from rasterio.warp import transform_bounds
from IPython.display import display, clear_output

# Shared helpers; workshop_utils.py sits beside this notebook.
from workshop_utils import (
    STAC_COLLECTION_URL, ITEM_ID, BAND_INFO, load_data,
    minmax, pstretch, to_uint8,
    plot_raster_distribution, interactive_raster_explorer,
    projected_plot_context, make_projected_image, apply_projected_axes,
)

pio.renderers.default = "plotly_mimetype+notebook_connected"

## 2. Where the data comes from

A **STAC** (SpatioTemporal Asset Catalog) catalogue describes geospatial data
in a standard way. The workshop *collection* contains one *item* per drone
survey, and each item lists its *assets*: the multispectral orthomosaic
(`ms`), the RGB orthomosaic, LiDAR products and so on. Each asset records a
URL plus metadata such as its band names and wavelengths.

In [ ]:
collection = pystac.Collection.from_file(STAC_COLLECTION_URL)
print(collection.title)
for survey in collection.get_items():
    print(f"  {survey.id}  ({survey.datetime:%d %b %Y})")

ms_asset = collection.get_item(ITEM_ID).assets["ms"]
print(f"\nThis chapter uses {ITEM_ID}: {ms_asset.title}")
print(f"URL:  {ms_asset.get_absolute_href()}")
print(f"Size: {ms_asset.extra_fields['file:size'] / 1e9:.1f} GB")

pd.DataFrame(ms_asset.extra_fields["bands"])[
    ["name", "description", "eo:center_wavelength", "eo:full_width_half_max"]
]

## 3. Load the study area

The full orthomosaic is several gigabytes, but it is stored as a
cloud-optimised GeoTIFF (COG), so we can read just the pixels inside our
study area. `load_data()` does this and returns an `xarray.Dataset` with
one variable per spectral band. NoData and background pixels (zero or
below) are converted to `NaN`.

In [ ]:
ds = load_data()
ds

Surface reflectance should lie between 0 and 1. Any value above 1 is
physically implausible, so we mask those pixels too.

In [ ]:
ds = ds.where(ds <= 1)
print(f"{ds.attrs['image_width_px']} x {ds.attrs['image_height_px']} pixels "
      f"at {ds.attrs['pixel_size_m'] * 100:.1f} cm, CRS {ds.attrs['crs']}")

## 4. Band specification and pixel statistics

In [ ]:
band_stats = pd.DataFrame([
    {
        "Band": band["var"], "Name": band["name"],
        "Centre (nm)": band["nm"], "FWHM (nm)": band["fwhm"],
        "Min": float(ds[band["var"]].min()),
        "Max": float(ds[band["var"]].max()),
        "Mean": float(ds[band["var"]].mean()),
        "Std": float(ds[band["var"]].std()),
        "Valid px": int(ds[band["var"]].count()),
    }
    for band in BAND_INFO
]).round(4)
band_stats

## 5. Sensor spectral response

The curves approximate each band as a Gaussian centred on its nominal
wavelength. The red-edge bands lie where vegetation reflectance changes
rapidly, making them useful for plant-condition monitoring.

In [ ]:
wl = np.linspace(420, 900, 1200)
fig_spec = go.Figure()

for band in BAND_INFO:
    sigma = band["fwhm"] / 2.355
    response = np.exp(-0.5 * ((wl - band["nm"]) / sigma) ** 2)
    red, green, blue = (int(band["colour"][i:i + 2], 16) for i in (1, 3, 5))
    fig_spec.add_trace(go.Scatter(
        x=wl, y=response, fill="tozeroy",
        fillcolor=f"rgba({red}, {green}, {blue}, 0.18)",
        line=dict(color=band["colour"], width=2),
        name=f"{band['name']} ({band['nm']} nm)",
        hovertemplate=(
            f"<b>{band['name']}</b><br>Centre: {band['nm']} nm"
            f"<br>FWHM: {band['fwhm']} nm<br>Response: %{{y:.3f}}<extra></extra>"
        ),
    ))

fig_spec.update_layout(
    title="MicaSense RedEdge-P Dual — Spectral Band Positions",
    xaxis_title="Wavelength (nm)",
    yaxis_title="Normalised Spectral Response",
    yaxis_range=[0, 1.2],
    height=420,
    legend=dict(font=dict(size=10), orientation="h", y=-0.25),
)
fig_spec.show()

## 6. True-colour composites

A true-colour image places the red, green and blue bands in the red, green
and blue display channels. Reflectance values are small, so each band must be
stretched to the 0–1 display range. A simple **min–max** stretch uses the
darkest and brightest pixels; a **percentile** stretch (2nd–98th) ignores a
few extreme pixels and usually gives much better contrast.

In [ ]:
rgb_bands = ["red668", "green560", "blue"]
rgb_minmax = np.dstack([minmax(ds[name].values) for name in rgb_bands])
rgb_pct = np.dstack([pstretch(ds[name].values, 2, 98) for name in rgb_bands])

# Map extent in projected coordinates (outer pixel edges).
half = ds.attrs["pixel_size_m"] / 2
left, right = float(ds.x.min()) - half, float(ds.x.max()) + half
bottom, top = float(ds.y.min()) - half, float(ds.y.max()) + half

fig, axes = plt.subplots(1, 2, figsize=(14, 6.5))
for ax, image, title in zip(
    axes, (rgb_minmax, rgb_pct),
    ("RGB (min–max stretch)", "RGB (2nd–98th percentile stretch)"),
):
    ax.imshow(image, extent=[left, right, bottom, top])
    ax.set_title(title)
    ax.set_xlabel("Easting (m)")
    ax.ticklabel_format(useOffset=False, style="plain")
axes[0].set_ylabel("Northing (m)")
plt.tight_layout()
plt.show()

## 7. Interactive band explorer

Each band can be viewed as a greyscale image beside a histogram of its
reflectance values. Here is the Red-650 band:

In [ ]:
band_plot_info = {
    band["var"]: {
        "label": f"{band['name']} ({band['nm']} nm)",
        "colorscale": "Gray",
        "histogram_color": band["colour"],
        "value_label": "Surface reflectance",
    }
    for band in BAND_INFO
}

fig_red650 = plot_raster_distribution(
    ds,
    variable="red650",
    metadata=band_plot_info,
    robust=False,
    statistics=("min", "median", "max"),
    title="Red-650 Band and Histogram",
)

Choose any band from the dropdown to redraw its image and histogram.

In [ ]:
band_selector, band_output = interactive_raster_explorer(
    ds,
    variables=[band["var"] for band in BAND_INFO],
    metadata=band_plot_info,
    default="red668",
    dropdown_description="Band:",
    robust=False,
    bins=80,
    statistics=("mean", "p2", "p98"),
    width=1200,
    height=570,
)

## 8. Build your own composite

Any three bands can be placed in the red, green and blue display channels.
Composites that use bands outside the visible range are called
**false-colour** images. Choose the bands and the percentile clip limits
below; the image redraws whenever a control changes.

In [ ]:
band_options = [(f"{band['name']} ({band['nm']} nm)", band["var"]) for band in BAND_INFO]
sel_r = widgets.Dropdown(options=band_options, value="red668", description="Red:")
sel_g = widgets.Dropdown(options=band_options, value="green560", description="Green:")
sel_b = widgets.Dropdown(options=band_options, value="blue", description="Blue:")
stretch_low = widgets.FloatSlider(value=2, min=0, max=15, step=0.5,
                                  description="Lower clip (%):", continuous_update=False)
stretch_high = widgets.FloatSlider(value=98, min=85, max=100, step=0.5,
                                   description="Upper clip (%):", continuous_update=False)
custom_output = widgets.Output()
context = projected_plot_context(ds)


def render_custom(change=None):
    low, high = stretch_low.value, stretch_high.value
    bands = (sel_r.value, sel_g.value, sel_b.value)
    rgb = to_uint8(np.dstack([pstretch(ds[name].values, low, high) for name in bands]))

    fig = go.Figure(make_projected_image(ds, rgb, context=context))
    apply_projected_axes(fig, context)
    fig.update_layout(
        title=f"Custom RGB — R={bands[0]}, G={bands[1]}, B={bands[2]}; P{low:g}–P{high:g}",
        template="plotly_white", width=900, height=650,
        margin=dict(l=90, r=40, t=80, b=80),
    )
    with custom_output:
        clear_output(wait=True)
        fig.show(config={"displaylogo": False})


for control in (sel_r, sel_g, sel_b, stretch_low, stretch_high):
    control.observe(render_custom, names="value")

display(widgets.VBox([
    widgets.HBox([sel_r, sel_g, sel_b]),
    widgets.HBox([stretch_low, stretch_high]),
    custom_output,
]))
render_custom()

## 9. Scene-wide spectral signature

Plotting each band's mean reflectance against its wavelength gives the
average spectrum of the study area. The shaded band shows ±1 standard
deviation.

In [ ]:
wavelengths = band_stats["Centre (nm)"].to_numpy()
means = band_stats["Mean"].to_numpy()
stds = band_stats["Std"].to_numpy()

fig_sig = go.Figure()
fig_sig.add_trace(go.Scatter(
    x=np.concatenate([wavelengths, wavelengths[::-1]]),
    y=np.concatenate([means + stds, (means - stds)[::-1]]),
    fill="toself", fillcolor="rgba(39,128,227,0.12)",
    line=dict(color="rgba(0,0,0,0)"), name="±1 standard deviation",
))
fig_sig.add_trace(go.Scatter(
    x=wavelengths, y=means, mode="lines+markers", text=band_stats["Name"],
    line=dict(color="#2780e3", width=2.5), marker=dict(size=8),
    hovertemplate="<b>%{text}</b><br>λ=%{x} nm<br>Mean=%{y:.4f}<extra></extra>",
    name="Scene mean",
))
fig_sig.update_layout(
    title="Scene-Wide Spectral Signature — All 10 Bands",
    xaxis_title="Wavelength (nm)", yaxis_title="Mean surface reflectance",
    height=400,
)
fig_sig.show()

## 10. Drone image on a basemap

The true-colour composite is reprojected to latitude/longitude and laid over
OpenStreetMap and Esri satellite imagery. A reduced-resolution preview keeps
the map responsive; use the layer control to switch basemaps.

In [ ]:
# Every 2nd pixel is plenty for an overview.
preview = to_uint8(rgb_pct[::2, ::2])

west, south, east, north = transform_bounds(
    ds.attrs["crs"], "EPSG:4326", left, bottom, right, top,
)

map_rgb = folium.Map(
    location=[(south + north) / 2, (west + east) / 2],
    zoom_start=19,
    max_zoom=23,
)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery",
    name="Esri Satellite",
).add_to(map_rgb)
folium.raster_layers.ImageOverlay(
    image=preview,
    bounds=[[south, west], [north, east]],
    opacity=0.88,
    name="Drone true-colour",
).add_to(map_rgb)
folium.LayerControl(collapsed=False).add_to(map_rgb)
map_rgb

## Activities

1. Use the band explorer from Blue through NIR-842. Which band makes
   vegetation brightest, and does this match the spectral signature?
2. Recreate CIR with Red=NIR-842, Green=Red-668, Blue=Green-531.
3. Toggle OpenStreetMap and Esri Satellite and identify features that
   align with the drone composite.

Percentile stretching improves display contrast by preventing a few
extreme pixels from compressing the useful tonal range.